In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch

from src.metric import *

In [ ]:
from torch.utils.data import DataLoader

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    util
)

from sentence_transformers.cross_encoder import CrossEncoder



In [ ]:
with open('../data/cleaned/clean_train_df.pkl','rb') as f:
    train_df=pickle.load(f)
    
with open('../data/cleaned/clean_val_df.pkl','rb') as f:
    val_df=pickle.load(f)
        
with open('../data/cleaned/clean_test_df.pkl','rb') as f:
    test_df=pickle.load(f)
    


In [ ]:
label_to_score={0:0,1:0.5,2:1.0}

In [ ]:
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
bi_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

bi_train_dataloader=DataLoader(bi_train_examples,shuffle=True,batch_size=64)

bi_train_loss=losses.CoSENTLoss(bi_encoder)

In [ ]:
epochs=1
bi_encoder.fit(
    train_objectives=[(bi_train_dataloader, bi_train_loss)],
    epochs=1,
    warmup_steps=int(len(bi_train_dataloader) *epochs* 0.1),
    show_progress_bar=True)


In [ ]:
unique_resume=train_df['resume_text'].drop_duplicates().tolist()

resume_embs=bi_encoder.encode(unique_resume,convert_to_tensor=True,
                             batch_size=64,show_progress_bar=True)
pos_rows=train_df[train_df['label']==2]

In [ ]:
hard_neg_rows=[]
top_k_hard_neg=20

for _,row in pos_rows.iterrows():
    jd=row['job_description_text']
    pos_resume=row['resume_text']

    jd_emb=bi_encoder.encode(jd,convert_to_tensor=True,batch_size=64)

    hits=util.semantic_search(jd_emb,resume_embs,top_k=top_k_hard_neg)
    hits=hits[0]

    added=0
    for hit in hits:
        idx=hit['corpus_id']
        candidate_resume=unique_resume[idx]
        if candidate_resume==pos_resume:
            continue
            
        existing=train_df[(train_df["job_description_text"] == jd)
            &
            (train_df["resume_text"] == candidate_resume)]
        
        if len(existing)>0 and existing.iloc[0]['label']>0:
            continue

        hard_neg_rows.append({
            "job_description_text": jd,
            "resume_text": candidate_resume,
            "label": 0
        })

        added+=1
        if added>3:
            break

hard_neg_df=pd.DataFrame(hard_neg_rows)
print(len(hard_neg_df))

In [ ]:
enhanced_train=pd.concat([train_df,hard_neg_df])
print(len(enhanced_train))

In [ ]:
#Retrain bi-encoder with enhanced data

In [ ]:
bi_examples_hard=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(enhanced_train['resume_text'],
                     enhanced_train['job_description_text'],enhanced_train['label'])
]

bi_dataloader_hard=DataLoader(bi_examples_hard,shuffle=True,batch_size=64)


In [ ]:
epochs = 2
best_score = float('-inf')
min_delta=0.01

for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}----------")
    
    bi_encoder.fit(
        train_objectives=[(bi_dataloader_hard, bi_train_loss)],
        epochs=1,
        warmup_steps=int(len(bi_dataloader_hard)* 0.1),
        show_progress_bar=True
    )

    val_resume_emb=bi_encoder.encode(
        val_df['resume_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )
    val_jd_emb =bi_encoder.encode(
        val_df['job_description_text'].tolist(),
        batch_size=64,convert_to_tensor=True,show_progress_bar=False
    )

    scores = torch.cosine_similarity(val_resume_emb, val_jd_emb).cpu().numpy()

    metrics = model_evaluation(scores, val_df, 'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])

    final_score = (0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +
                   0.1*metrics['mrr_score'])

    if final_score > best_score+min_delta:
        best_score=final_score
        bi_encoder.save("../models/hard_negative/bi_encoder")
        
        


In [ ]:
test_resume_emb=bi_encoder.encode(test_df['resume_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)
test_jd_emb = bi_encoder.encode(test_df['job_description_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(test_resume_emb,test_jd_emb).cpu().numpy()
metrics = model_evaluation(scores, test_df, 'job_description_text')
print(metrics)

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",num_labels=1)

In [ ]:
cross_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(enhanced_train['resume_text'],
                     enhanced_train['job_description_text'],enhanced_train['label'])
]

cross_dataloader=DataLoader(cross_examples,shuffle=True,batch_size=64)


In [ ]:
test_resume_emb=bi_encoder.encode(test_df['resume_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)
test_jd_emb = bi_encoder.encode(test_df['job_description_text'].tolist(),
                              batch_size=64,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(test_resume_emb,test_jd_emb).cpu().numpy()
metrics = model_evaluation(scores, test_df, 'job_description_text')
print(metrics)

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",num_labels=1)

In [ ]:
cross_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(enhanced_train['resume_text'],
                     enhanced_train['job_description_text'],enhanced_train['label'])
]

cross_dataloader=DataLoader(cross_examples,shuffle=True,batch_size=64)


In [ ]:
epochs=2
best_score=float('-inf')
min_delta=0.01
patience=2
count=0

for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder.fit(train_dataloader=cross_dataloader,epochs=1,
                             warmup_steps=int(len(cross_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder.predict(val_pairs,batch_size=64,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])    

    final_score =(0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +0.1*metrics['mrr_score'])
    
    if final_score>best_score+0.01:
        best_score=final_score
        cross_encoder.save("../models/hard_negative/cross_encoder")
        count=0
    else:
        count+=1
        
    if count==patience:
        print("Early Stopping")
        break
    


val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder.predict(val_pairs,batch_size=64,show_progress_bar=False)

ranked_result=[]
val_df_new=val_df.copy()
val_df_new['score']=scores


In [ ]:
test_resume_texts=test_df['resume_text'].drop_duplicates().tolist()

test_resume_embs=bi_encoder.encode(test_resume_texts,convert_to_tensor=True,
                             batch_size=64,show_progress_bar=True)


In [ ]:
bi_ndcg_list = []
cross_ndcg_list= []

bi_spearman_list = []
cross_spearman_list = []

bi_topk_list = []
cross_topk_list = []

bi_map_list=[]
cross_map_list=[]

bi_mrr_list=[]
cross_mrr_list=[]

top_k=50

In [ ]:
for jd,group in test_df.groupby('job_description_text'):

    if len(group)<2:
        continue
        
    group_label=group[['resume_text','label']].drop_duplicates()
    label_map=dict(zip(group_label['resume_text'],group_label['label']))

    # Stage 1 — Bi-Encoder retrieval (top-50)

    jd_emb=bi_encoder.encode(jd,convert_to_tensor=True)
    
    hits=util.semantic_search(jd_emb,test_resume_embs, top_k=top_k)
    hits=hits[0]

    retrieval_rows = []
    for hit in hits:
        idx=hit["corpus_id"]
        resume_text=test_resume_texts[idx]
        if resume_text not in label_map:
            continue
        retrieval_rows.append({
            "resume_text":resume_text,
            "score": float(hit["score"]),
            "label": label_map[resume_text]
        })
    retrieval_df=pd.DataFrame(retrieval_rows)
    if retrieval_df.empty:
      continue

    #Ndcg
    bi_ndcg=ndcg_metric(retrieval_df)
    if bi_ndcg is not None:
        bi_ndcg_list.append(bi_ndcg)
    
    # Spearman
    bi_corr=corr_metric(retrieval_df)
    if bi_corr is not None:
        bi_spearman_list.append(bi_corr)

    # Top-K accuracy
    bi_topk=topk_metric(retrieval_df)
    if bi_topk is not None:
        bi_topk_list.append(bi_topk)

    #Mrr
    bi_mrr=mrr_metric(retrieval_df)
    if bi_mrr is not None:
        bi_mrr_list.append(bi_mrr)
    
    #Mrr
    bi_map=map_metric(retrieval_df)
    if bi_map is not None:
        bi_map_list.append(bi_map)
    
            
    # Stage 2 — Cross-Encoder reranking (on bi-encoder top-50)

    pairs=list(zip([jd]*len(retrieval_df),retrieval_df['resume_text']))

    cross_scores=cross_encoder.predict(pairs,batch_size=64)

    rerank_df=retrieval_df.copy()
    rerank_df['score']=cross_scores
    rerank_df=rerank_df.sort_values("score",ascending=False).reset_index(drop=True)


    #Ndcg
    cross_ndcg=ndcg_metric(rerank_df)
    if cross_ndcg is not None:
        cross_ndcg_list.append(cross_ndcg)
    
    # Spearman
    cross_corr=corr_metric(rerank_df)
    if cross_corr is not None:
        cross_spearman_list.append(cross_corr)

    # Top-K accuracy
    cross_topk=topk_metric(rerank_df)
    if cross_topk is not None:
        cross_topk_list.append(cross_topk)

    #Mrr
    cross_mrr=mrr_metric(rerank_df)
    if cross_mrr is not None:
        cross_mrr_list.append(cross_mrr)
    
    #Mrr
    cross_map=map_metric(rerank_df)
    if cross_map is not None:
        cross_map_list.append(cross_map)
        
            
    
print("\n" + "=" * 60)
print("FINAL TWO-STAGE RETRIEVAL RESULTS")

print("=" * 45)
print(f"{'Metric':<25} {'Bi-Encoder':>8}   {'Cross-Encoder':>13}")
print("=" * 45)
print(f"{'NDCG@10':<25} {np.mean(bi_ndcg_list):>8.4f}   {np.mean(cross_ndcg_list):>13.4f}")
print(f"{'Spearman ':<25} {np.mean(bi_spearman_list):>8.4f}   {np.mean(cross_spearman_list):>13.4f}")
print(f"{'Top-{3} Accuracy':<25} {np.mean(bi_topk_list):>8.4f}   {np.mean(cross_topk_list):>13.4f}")
print(f"{'MRR':<25} {np.mean(bi_mrr_list):>8.4f}   {np.mean(cross_mrr_list):>13.4f}")
print(f"{'MAP':<25} {np.mean(bi_map_list):>8.4f}   {np.mean(cross_map_list):>13.4f}")
print("=" * 45)

    
    
